RTE contain only train and validation data, its test data has no label, we combine and split the dataset.

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import datasets
from sklearn.model_selection import train_test_split

In [ ]:
# Function to filter out data without labels
def filter_dataset(dataset, valid_label_list):
    return dataset.filter(lambda x: x['label'] in valid_label_list)

In [ ]:
# Function to generate train, valid, test splits (8:1:1) with shuffling
def create_splits(dataset, test_size=0.1, valid_size=0.1, random_state=42):
    # Convert to DataFrame for easier manipulation
    df = dataset.to_pandas()

    # Split the data into train+valid and test; by default the data will be shuffled before splitting
    train_valid_df, test_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df['label'])
    
    # Split the train+valid data into train and valid
    train_df, valid_df = train_test_split(train_valid_df, test_size=valid_size/(1-test_size), random_state=random_state, stratify=train_valid_df['label'])

    # Convert back to datasets.Dataset
    return DatasetDict({
        'train': Dataset.from_pandas(train_df.reset_index(drop=True)),
        'validation': Dataset.from_pandas(valid_df.reset_index(drop=True)),
        'test': Dataset.from_pandas(test_df.reset_index(drop=True))
    })

In [ ]:
# Load the dataset from HuggingFace and define valid labels
dataset = load_dataset("nyu-mll/glue", "rte")
valid_labels = [0,1] # 0: entailment, 1: not entailment

# Check the available splits and process accordingly
if 'train' in dataset and 'validation' in dataset and 'test' in dataset:
    # Check if test split has labels
    # print(dataset['test'].filter(lambda x: x['label'] in valid_labels))
    if dataset['test'].filter(lambda x: x['label'] in valid_labels).num_rows > 0:
        print("The dataset has complete train/valid/test data, we only filter out invalid data")
        # Filter out data without labels
        dataset = DatasetDict({
            'train': filter_dataset(dataset['train'], valid_labels),
            'validation': filter_dataset(dataset['validation'], valid_labels),
            'test': filter_dataset(dataset['test'], valid_labels),
        })
    # Generate new splits from train and valid (abort test)
    else:
        print("The dataset has no label for test data, we generate train/valid/test data from train/valid data")
        combined_dataset = datasets.concatenate_datasets([dataset['train'], dataset['validation']]).filter(lambda x: x['label'] in valid_labels)
        dataset = create_splits(combined_dataset)
# Generate new splits from train and test (no validation set available)
elif 'train' in dataset and 'test' in dataset:
    print("The dataset does not have validation data, we generate train/valid/test data from train/test data")
    combined_dataset = datasets.concatenate_datasets([dataset['train'], dataset['test']]).filter(lambda x: x['label'] in valid_labels)
    dataset = create_splits(combined_dataset)

for split in dataset.keys():
    if "idx" in dataset[split].column_names:
        dataset[split] = dataset[split].remove_columns("idx")

In [ ]:
# examine the processed dataset
for split in dataset.keys():
    print(f"Split: {split}")
    print(f"Number of examples: {len(dataset[split])}")
    print(dataset[split].features)

In [ ]:
# Rename columns for all splits
def rename_columns(dataset, column_mapping):
    return dataset.rename_columns(column_mapping)

# Define the column mapping
column_mapping = {
    "sentence1": "premise",
    "sentence2": "hypothesis"
}

# Rename the columns for all splits
dataset = {split: rename_columns(split_data, column_mapping) for split, split_data in dataset.items()}

# Inspect the result
print(dataset)

In [ ]:
from datasets import DatasetDict
dataset = DatasetDict(dataset)

In [ ]:
dataset.push_to_hub("rte")